# RETFound-Green contra o R12 (F10)

Ajusta o RETFound-Green (ViT-S/14 pré-treinada em 75 mil fotos de retina; Engelmann e Bernabeu,
arXiv 2405.00117) para a mesma tarefa do R12, com os mesmos dados de treino (EyePACS, extras de 2015,
DDR, APTOS, DeepDRiD e IDRiD), e o mede na mesma validação. Em PyTorch: este notebook não carrega
TensorFlow. As imagens são as dos caches de 768 px (com Ben), reduzidas para os 392 px da rede.

**Regra fixada antes de rodar:** o RETFound-Green ganha do R12 semente 42 se a especificidade por
paciente com TTA2 na validação for maior, com o intervalo de 95% da diferença (bootstrap pareado de
pacientes) acima de zero. O teste do EyePACS fica fechado.

**Licença:** os pesos só podem ser usados em pesquisa não comercial, sem projeto com empresa envolvida.
Nada deste modelo, nem um derivado dele, vai para o app dos postos sem permissão escrita do autor.

A lógica está em `src/retinopatia/retfound.py`. Sessão A100 com RAM alta (~1 h). Na mesma sessão, primeiro
uma fumaça de poucos minutos: marque `FUMACA`, desmarque `DESLIGAR_AO_TERMINAR` e execute tudo. Se passar,
reinicie a sessão (Ambiente de execução → Reiniciar sessão; a configuração é lida no import), desmarque
`FUMACA`, marque `DESLIGAR_AO_TERMINAR` e execute tudo de novo. A sessão desliga sozinha no fim.


## Preparação do repositório


In [ ]:
# O código do projeto vive no repositório do GitHub.
# Esta célula traz (ou atualiza) esse código dentro do Colab.

import os
import sys

URL_REPOSITORIO = "https://github.com/Rcorrea07/ML_Retinopatia.git"
PASTA_REPOSITORIO = "/content/ML_Retinopatia"

# Versão do código: "origin/main" é a mais recente; uma branch
# ("origin/refatoracao"), uma tag ou um commit reproduz outra. O commit de
# cada treino fica no configuracao.json da pasta de resultados.
REVISAO_CODIGO = "origin/main"  #@param {type:"string"}

# Experimento (src/retinopatia/experimentos.py): o RETFound-Green com os
# dados de treino do R12. Para trocar depois, reinicie o ambiente.
EXPERIMENTO = "F10_retfound_green"  #@param {type:"string"}
os.environ["RETINOPATIA_EXPERIMENTO"] = EXPERIMENTO

# Fumaça: poucos pacientes e uma época por fase, para conferir em minutos
# que tudo roda e salva. Métricas não valem e não há comparação com o R12.
FUMACA = False  #@param {type:"boolean"}
os.environ["RETINOPATIA_FUMACA"] = "1" if FUMACA else "0"

# O teste do EyePACS já foi gasto no R09: fica fechado
os.environ["RETINOPATIA_AVALIAR_TESTE"] = "0"

# Semente do treino (a divisão dos pacientes não muda)
SEMENTE = 42  #@param {type:"integer"}
os.environ["RETINOPATIA_SEED_TREINO"] = str(SEMENTE)

# True = encerra a sessão no fim, também se der erro
DESLIGAR_AO_TERMINAR = True  #@param {type:"boolean"}

if not os.path.exists(PASTA_REPOSITORIO):
    !git clone {URL_REPOSITORIO} {PASTA_REPOSITORIO}

!git -C {PASTA_REPOSITORIO} fetch --tags origin
!git -C {PASTA_REPOSITORIO} checkout --detach {REVISAO_CODIGO}
!git -C {PASTA_REPOSITORIO} log -1 --date=short --format="Código no commit %h (%cd): %s"

# Torna o pacote "retinopatia" importável neste notebook
caminho_src = os.path.join(PASTA_REPOSITORIO, "src")

if caminho_src not in sys.path:
    sys.path.insert(0, caminho_src)

# Recarrega os módulos automaticamente quando esta célula traz código
# novo, evitando ter que reiniciar o ambiente a cada edição no VS Code.
# O Colab usa Python 3.12+, que não tem mais o módulo "imp", mas a
# versão do IPython instalada nele ainda o importa no autoreload.
# Um substituto mínimo resolve; se mesmo assim falhar, o notebook
# segue sem autoreload (aí é preciso reiniciar após cada git pull).
import importlib
import types

try:
    import imp
except ModuleNotFoundError:
    sys.modules["imp"] = types.ModuleType("imp")
    sys.modules["imp"].reload = importlib.reload

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception as erro:
    print("Autoreload indisponível:", erro)

print("Repositório pronto em:", PASTA_REPOSITORIO)

## PyTorch, timm e Drive


In [ ]:
import importlib.util

# O Colab já traz o PyTorch; o timm costuma vir também
if importlib.util.find_spec("timm") is None:
    !pip install -q timm

import timm
import torch

print("PyTorch", torch.__version__, "| timm", timm.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma")

from retinopatia import ambiente_colab, retfound

ambiente_colab.montar_drive()


## Treino, validação e comparação com o R12


In [ ]:
resultado = retfound.rodar(
    desligar_ao_terminar=DESLIGAR_AO_TERMINAR
)
